# Import Library

ในขั้นตอนนี้จะนำเข้าไลบรารีที่จำเป็นสำหรับการวิเคราะห์ข้อมูล
สร้างโมเดล Machine Learning และประเมินผลลัพธ์

In [484]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA

from sklearn.linear_model import LinearRegression
from sklearn.linear_model import LogisticRegression

from sklearn.metrics import (
    mean_absolute_error,
    mean_squared_error,
    r2_score,
    accuracy_score,
    confusion_matrix,
    classification_report,
    roc_curve,
    auc
)

# Load Dataset

โหลดชุดข้อมูล Red Wine Quality จากไฟล์ CSV
เพื่อเตรียมข้อมูลสำหรับการวิเคราะห์และสร้างโมเดล

In [ ]:
df = pd.read_csv("dataset.csv")

df.head()

FileNotFoundError: [Errno 2] No such file or directory: 'winequality-red.csv'

# Dataset Overview

ตรวจสอบข้อมูลเบื้องต้นของชุดข้อมูล ได้แก่

- จำนวนข้อมูล
- ชนิดข้อมูล
- ค่าสถิติพื้นฐาน
- Missing Value

In [ ]:
print("Shape :", df.shape)

print("\nInformation")
df.info()

print("\nMissing Value")
print(df.isnull().sum())

print("\nStatistics")
display(df.describe())

# Data Exploration (EDA)

สำรวจการกระจายตัวของข้อมูลด้วย Histogram
เพื่อดูแนวโน้มของข้อมูลแต่ละตัวแปร

In [ ]:
df.hist(figsize=(14,10))

plt.tight_layout()

plt.show()

# LAB 1 : Regression

Regression คือการทำนายค่าตัวเลข (Continuous Value)

ในตัวอย่างนี้จะทำนาย **Quality ของไวน์**
โดยเปรียบเทียบระหว่าง

- Simple Linear Regression
- Multiple Linear Regression

## Simple Linear Regression

ใช้ตัวแปรอิสระเพียง **1 ตัว**
คือ `alcohol`

เพื่อทำนายค่า `quality`

In [ ]:
# Feature และ Target

X_simple = df[["alcohol"]]

y = df["quality"]

X_train, X_test, y_train, y_test = train_test_split(
    X_simple,
    y,
    test_size=0.2,
    random_state=42
)

In [ ]:
model_simple = LinearRegression()

model_simple.fit(X_train, y_train)

y_pred_simple = model_simple.predict(X_test)

## ประเมินผล Simple Linear Regression

ใช้ตัวชี้วัด

- MAE
- RMSE
- R² Score

In [ ]:
mae = mean_absolute_error(y_test, y_pred_simple)

rmse = np.sqrt(mean_squared_error(y_test, y_pred_simple))

r2 = r2_score(y_test, y_pred_simple)

print("Simple Linear Regression")
print("MAE :", round(mae,3))
print("RMSE :", round(rmse,3))
print("R² :", round(r2,3))

In [ ]:
plt.figure(figsize=(6,4))

plt.scatter(
    X_test,
    y_test,
    color="blue",
    label="Actual"
)

plt.plot(
    X_test,
    y_pred_simple,
    color="red",
    linewidth=2,
    label="Prediction"
)

plt.xlabel("Alcohol")

plt.ylabel("Quality")

plt.title("Simple Linear Regression")

plt.legend()

plt.show()

# Multiple Linear Regression

ใช้ตัวแปรทุกคอลัมน์
(ยกเว้น Quality)

เพื่อเพิ่มความแม่นยำในการทำนาย

In [ ]:
X = df.drop("quality", axis=1)

y = df["quality"]

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42
)

In [ ]:
model_multiple = LinearRegression()

model_multiple.fit(X_train, y_train)

y_pred_multiple = model_multiple.predict(X_test)

## ประเมินผล Multiple Linear Regression

เปรียบเทียบผลลัพธ์กับ
Simple Linear Regression

In [ ]:
mae2 = mean_absolute_error(y_test, y_pred_multiple)

rmse2 = np.sqrt(mean_squared_error(y_test, y_pred_multiple))

r22 = r2_score(y_test, y_pred_multiple)

print("Multiple Linear Regression")
print("MAE :", round(mae2,3))
print("RMSE :", round(rmse2,3))
print("R² :", round(r22,3))

In [ ]:
result = pd.DataFrame({
    "Actual": y_test.values,
    "Predicted": y_pred_multiple
})

result.head()

## เปรียบเทียบผล Regression

เปรียบเทียบค่า MAE, RMSE และ R²
ของทั้งสองโมเดล

In [ ]:
compare = pd.DataFrame({
    "Model": [
        "Simple Linear Regression",
        "Multiple Linear Regression"
    ],
    "MAE": [
        mae,
        mae2
    ],
    "RMSE": [
        rmse,
        rmse2
    ],
    "R²": [
        r2,
        r22
    ]
})

compare

# LAB 2 : Classification

Classification คือการจำแนกข้อมูลออกเป็นกลุ่ม

ในตัวอย่างนี้จะจำแนกคุณภาพไวน์เป็น

- Good Wine (1)
- Bad Wine (0)

โดยใช้ Logistic Regression

## Preparing Classification Data

สร้างคอลัมน์ใหม่ชื่อ `quality_label`

- Quality >= 6 = Good Wine (1)
- Quality < 6 = Bad Wine (0)

In [ ]:
df["quality_label"] = (df["quality"] >= 6).astype(int)

df[["quality","quality_label"]].head()

## Feature และ Target

เลือกตัวแปรสำหรับฝึกโมเดล

Target คือ quality_label

In [ ]:
X = df.drop(["quality","quality_label"], axis=1)

y = df["quality_label"]

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42
)

## Feature Scaling

ใช้ StandardScaler เพื่อปรับข้อมูลให้อยู่ในช่วงใกล้เคียงกัน

ช่วยให้โมเดลเรียนรู้ได้ดีขึ้น

In [ ]:
scaler = StandardScaler()

X_train_scaled = scaler.fit_transform(X_train)

X_test_scaled = scaler.transform(X_test)

## Principal Component Analysis (PCA)

ลดจำนวนคุณลักษณะให้เหลือ 2 มิติ

เพื่อใช้แสดงผลข้อมูลและ Decision Boundary

In [ ]:
pca = PCA(n_components=2)

X_train_pca = pca.fit_transform(X_train_scaled)

X_test_pca = pca.transform(X_test_scaled)

print(X_train_pca.shape)

## Logistic Regression

สร้างโมเดล Logistic Regression
เพื่อจำแนก Good Wine และ Bad Wine

In [ ]:
model = LogisticRegression(random_state=42)

model.fit(X_train_pca, y_train)

y_pred = model.predict(X_test_pca)

## Accuracy

ตรวจสอบความถูกต้องของโมเดล

In [ ]:
acc = accuracy_score(y_test, y_pred)

print("Accuracy :", round(acc,3))

## Confusion Matrix

แสดงจำนวนข้อมูลที่ทำนายถูกและผิด

In [ ]:
cm = confusion_matrix(y_test, y_pred)

print(cm)

In [ ]:
plt.figure(figsize=(5,4))

plt.imshow(cm)

plt.title("Confusion Matrix")

plt.colorbar()

plt.xticks([0,1],["Bad","Good"])

plt.yticks([0,1],["Bad","Good"])

plt.xlabel("Predicted")

plt.ylabel("Actual")

plt.show()

## Classification Report

แสดงผลการประเมินโมเดล ได้แก่

- Precision
- Recall
- F1-score

In [ ]:
print(classification_report(y_test, y_pred))

## ROC Curve

ROC Curve ใช้ประเมินความสามารถในการจำแนกข้อมูลของโมเดล

ยิ่งค่า AUC เข้าใกล้ 1 ยิ่งแสดงว่าโมเดลมีประสิทธิภาพดี

In [ ]:
y_prob = model.predict_proba(X_test_pca)[:,1]

fpr, tpr, _ = roc_curve(y_test, y_prob)

roc_auc = auc(fpr, tpr)

plt.figure(figsize=(6,5))

plt.plot(fpr, tpr, label=f"AUC = {roc_auc:.3f}")

plt.plot([0,1],[0,1],"--")

plt.xlabel("False Positive Rate")

plt.ylabel("True Positive Rate")

plt.title("ROC Curve")

plt.legend()

plt.show()

## Decision Boundary Visualization

แสดงการแบ่งพื้นที่การจำแนกข้อมูลของ Logistic Regression
บนข้อมูลที่ลดมิติด้วย PCA เหลือ 2 มิติ

In [ ]:
plt.figure(figsize=(7,5))

plt.scatter(
    X_train_pca[:,0],
    X_train_pca[:,1],
    c=y_train,
    cmap="coolwarm"
)

plt.xlabel("Principal Component 1")

plt.ylabel("Principal Component 2")

plt.title("PCA Visualization")

plt.show()

# LAB 3 : Model Comparison

ในส่วนนี้จะเปรียบเทียบประสิทธิภาพของโมเดล Regression และ Classification

หัวข้อที่ศึกษา

- Simple vs Multiple Linear Regression
- Training vs Testing Performance
- Regression vs Classification
- Model Performance Metrics

## เปรียบเทียบ Simple และ Multiple Linear Regression

เปรียบเทียบค่า MAE, RMSE และ R²
เพื่อดูว่าโมเดลใดให้ผลการทำนายดีกว่า

In [ ]:
comparison_regression = pd.DataFrame({
    "Model": [
        "Simple Linear Regression",
        "Multiple Linear Regression"
    ],
    "MAE": [
        mae,
        mae2
    ],
    "RMSE": [
        rmse,
        rmse2
    ],
    "R² Score": [
        r2,
        r22
    ]
})

comparison_regression

## Training และ Testing Performance

เปรียบเทียบผลการทำนายของข้อมูล Train และ Test

หากค่าทั้งสองใกล้เคียงกัน
แสดงว่าโมเดลสามารถเรียนรู้ข้อมูลได้ดี

In [ ]:
train_score = model_multiple.score(X_train, y_train)

test_score = model_multiple.score(X_test, y_test)

print("Training Score :", round(train_score,3))

print("Testing Score :", round(test_score,3))

## Regression vs Classification

เปรียบเทียบตัวชี้วัดของทั้งสองโมเดล

Regression ใช้

- MAE
- RMSE
- R²

Classification ใช้

- Accuracy
- Precision
- Recall
- F1-score

In [ ]:
print("Regression")

print("MAE :", round(mae2,3))

print("RMSE :", round(rmse2,3))

print("R² :", round(r22,3))

print("\nClassification")

print("Accuracy :", round(acc,3))

## Model Performance Metrics

สรุปผลการประเมินของแต่ละโมเดล

In [ ]:
summary = pd.DataFrame({

    "Model":[
        "Simple Regression",
        "Multiple Regression",
        "Logistic Regression"
    ],

    "MAE":[
        round(mae,3),
        round(mae2,3),
        "-"
    ],

    "RMSE":[
        round(rmse,3),
        round(rmse2,3),
        "-"
    ],

    "R²":[
        round(r2,3),
        round(r22,3),
        "-"
    ],

    "Accuracy":[
        "-",
        "-",
        round(acc,3)
    ]

})

summary

# สรุปผลการทดลอง

จากการทดลองพบว่า

- Multiple Linear Regression ให้ผลการทำนายคุณภาพไวน์แม่นยำกว่า Simple Linear Regression เนื่องจากใช้ข้อมูลจากหลายคุณลักษณะร่วมกัน
- Logistic Regression สามารถจำแนกไวน์เป็น Good Wine และ Bad Wine ได้อย่างมีประสิทธิภาพ
- การใช้ PCA ช่วยลดจำนวนคุณลักษณะและทำให้สามารถแสดงผลข้อมูลในรูปแบบ 2 มิติได้
- การประเมินผลด้วย MAE, RMSE, R² และ Accuracy ช่วยให้สามารถเปรียบเทียบประสิทธิภาพของโมเดลได้อย่างชัดเจน

# Conclusion

ในการทดลองครั้งนี้ได้ศึกษาการสร้างโมเดล Machine Learning ทั้ง Regression และ Classification โดยใช้ชุดข้อมูล Red Wine Quality จาก Kaggle

ผลการทดลองแสดงให้เห็นว่า Multiple Linear Regression มีประสิทธิภาพดีกว่า Simple Linear Regression สำหรับการทำนายคุณภาพไวน์ ส่วน Logistic Regression สามารถจำแนกคุณภาพไวน์ออกเป็น Good Wine และ Bad Wine ได้อย่างเหมาะสม

การเตรียมข้อมูล การปรับขนาดข้อมูล (Feature Scaling) และการใช้ PCA เป็นขั้นตอนที่ช่วยให้การสร้างโมเดลมีประสิทธิภาพมากขึ้น และสามารถนำความรู้ที่ได้ไปประยุกต์ใช้กับชุดข้อมูลอื่น ๆ ในงาน Machine Learning ได้